# 02_4 Fine-tuning of the direct three-class classifier on synthetic vectors

**Overview.** This notebook repeats the tuning of `02_3_pca_vector_to_classifier_finetunning.ipynb` (same classes, same model, same folds, same search, selection and evaluation) with one change: the classifier is **fitted on synthetic bags** instead of real ones. The synthetic bags are those of `01_3_pca_to_synthetic_vectors.ipynb`: for every class, tile vectors are sampled from cluster-specific Gaussians estimated on the PCA scores of the official Training slides, and every bag blends the size and cluster composition of several real Training slides. The classes are Non-cancer (N, PB, UDH, FEA), non-invasive atypia/neoplasia (ADH and DCIS) and invasive carcinoma (IC). The question is how well a model that has never seen a real tile in its fit data classifies real slides, and how it compares with the models trained on real vectors.

Within every cross-validation fold the model is **fitted on the synthetic slides** (all of them, role Training) and its epoch is chosen on the **real early-stopping slides** of the fold (the same slides as in 02_3), so the early-stopping score measures the transfer to real tissue. The held-out slides of the fold and the official Testing slides are real and are never used for fitting or stopping. The number of clusters `syn_k` of the synthetic bags (the folder `k_<K>`) is a hyperparameter of the Bayesian optimisation, together with the learning rate, weight decay, dropout, weight of the coarse loss, instance-loss sample size and tile cap of 02_3.

**Scoring and leakage.** The generator of 01_3 saw *all* official Training slides, which are also in the held-out folds of the pooled cross-validation (Training + Validation). The out-of-fold score of a Training slide is therefore optimistic for a model fitted on its synthetic derivatives. The constant `SCORING` selects the score that drives the search: `'cv'` pools every held-out slide (comparable with 02_3, optimistic), `'validation'` uses only the out-of-fold official Validation slides of patients without Training slides, which the generator never saw (clean, but few slides, hence noisy). The Testing evaluation is unaffected: no Testing slide enters the PCA, the generator or any choice.

```mermaid
flowchart TD
    subgraph IN["Inputs"]
        PCA[("pca_DIMENSION/UNI2-h/<br/>real PCA vectors (01_2)")]
        SYN[("synthetic/gaussian/pca_DIMENSION/k_K/<br/>labels/BRACS_synthetic.csv (01_3)")]
        LAB[("labels/BRACS.xlsx<br/>class, patient, official set")]
        FULL[("level_1/UNI2-h/<br/>1536-d vectors, Testing slides only")]
        OLD[("classifier/final/<br/>models of 02_1, 02_2, 02_3")]
    end

    subgraph S3["3. Data"]
        D1["3.1 build_slide_table + load_bags<br/>real bags: stop, held-out, Testing"]
        D2["3.2 relabel_table<br/>Non-cancer / atypia-neoplasia / IC"]
        D3["3.3 make_cv_folds<br/>the folds of 02_3"]
        D4["3.4 load_synthetic_sets<br/>one set per k"]
        D5["3.5 validation_mask<br/>score of SCORING"]
    end

    subgraph S4["4. Bayesian optimisation"]
        B1["synthetic_objective_factory + run_study<br/>fit on synthetic bags of syn_k,<br/>stop on real stop slides, score on held-out"]
        B2["trials_table + plot_tuning"]
    end

    subgraph S5["5-7. Selection, confirmation, ensemble"]
        C1["select_setting + confirmation<br/>synthetic_cross_validate, 2 repeats"]
        N1["oof_by_repeat + single_vs_ensemble"]
    end

    subgraph S8["8. Testing set, used once"]
        E1["predict_ensemble<br/>real Testing slides"]
        E2["class_views + bootstrap_views"]
        E3["summary_across_sets + paired_bootstrap_views<br/>vs 02_3, 02_1 and 02_2"]
    end

    OUT[("classifier/final/noncancer_noninvasive_invasive_synpcaDIMENSION/<br/>ensemble_date-time.pt")]

    PCA --> D1
    LAB --> D1
    D1 --> D2 --> D3
    SYN --> D4
    D3 --> D5
    D3 --> B1
    D4 --> B1
    D5 --> B1
    B1 --> B2 --> C1 --> N1 --> E1 --> E2
    E1 --> E3
    OLD --> E3
    FULL --> E3
    N1 --> OUT
```

## 1. Setup

The environment is recorded so the run can be reproduced.

In [1]:
# STEP 1: Standard-library imports and general-purpose third-party libraries
import sys
import matplotlib
import numpy as np
import optuna
import pandas as pd
import sklearn
import torch

# STEP 2: Print library versions for reproducibility
print('System Version:', sys.version)
print('PyTorch version:', torch.__version__)
print('Scikit-learn version:', sklearn.__version__)
print('Optuna version:', optuna.__version__)
print('Numpy version:', np.__version__)
print('Pandas version:', pd.__version__)
print('Matplotlib version:', matplotlib.__version__)

System Version: 3.14.7 (main, Aug 10 2026, 07:46:56) [GCC 16.1.1 20260728]
PyTorch version: 2.15.0+cu132
Scikit-learn version: 1.9.1
Optuna version: 5.1.0
Numpy version: 2.5.4
Pandas version: 3.1.1
Matplotlib version: 3.11.2


The model is trained on the GPU when one is available and on the CPU otherwise (`utils.device.get_device`).

In [2]:
# STEP 1: Select the device (pass e.g. get_device("cpu") to force the CPU)
from utils.device import get_device
DEVICE = get_device()

# STEP 2: Report it
print('Device:', DEVICE, '|', torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else 'no GPU, running on the CPU')

Device: cuda | NVIDIA RTX PRO 4500 Blackwell


The project's own building blocks are imported here. The search, the selection rule and the ensemble file are in `utils/mil_tuning.py`; the synthetic-data folds, the scoring and the objective with the number of clusters are in `utils/synthetic_training.py`; the data, model, cross-validation and metric code is shared with notebooks 02 to 02_3.

In [3]:
# STEP 1: Standard-library imports
import json
from pathlib import Path

# STEP 2: Import this project's own building blocks
from utils.dataset_overview import build_slide_table
from utils.mil_data import load_bags
from utils.mil_cv import fold_summary, make_cv_folds
from utils.mil_train import device_description, load_classifier, predict, predict_ensemble, set_seed
from utils.classification_metrics import evaluate_predictions
from utils.final_classifier import (bootstrap_views, class_views, format_differences, format_views, paired_bootstrap_views, plot_final_results, relabel_table, summary_across_sets)
from utils.mil_tuning import (DEFAULT_SPACE, eligible_trials, load_ensemble, oof_by_repeat, plot_tuning, run_study, save_ensemble, score_cv, select_setting, setting_kwargs, single_vs_ensemble,
                              trials_table)
from utils.synthetic_training import (load_synthetic_sets, score_oof, setting_with_k, subset_cv, synthetic_cross_validate, synthetic_objective_factory, synthetic_tile_counts, validation_mask)

## 2. Constants

Every setting is defined here. `DIMENSION` is the number of PCA components: the real vectors are read from `pca_<DIMENSION>/` (notebook 01_2) and the synthetic bags from `SYNTHETIC_ROOT/<SYNTHETIC_METHOD>/pca_<DIMENSION>/k_<K>/` (notebook 01_3, which must have been run with that dimension and with every `K_VALUES` entry); the label sheet of the synthetic slides is `SYNTHETIC_LABELS_FILE`. `CACHE_IN_RAM` keeps every synthetic bag that was read in memory as float16 (about tiles x `DIMENSION` x 2 bytes per bag, for all four sets together), because the bags are read in every epoch of every trial; set it to False to memory-map the files instead. `FEATURES_FULL_DIR` holds the original 1536-dimensional vectors, read only for the Testing slides of the comparison with notebooks 02_1 and 02_2.

`SCORING` is `'cv'` or `'validation'` (see the overview). The searched parameters and ranges are `SPACE`, the search space of 02_3 extended by `syn_k`, the number of clusters of the synthetic bags; the first trial is `BASELINE` (the setting of notebook 02_1 and the middle value of `K_VALUES`). Class weighting, tile dropout, the weight of the slide loss, the number of epochs and the patience are fixed as in 02_3, and so are the recall floor `MIN_CLASS_RECALL`, `N_TRIALS`, `N_CONFIRM` and `REPEATS`. The study is named after the dimension and the scoring, so studies for different dimensions and scorings are kept apart.

In [4]:
# Major variables
DIMENSION = 256                                                         # STEP 1: Number of PCA components (needs pca_<DIMENSION>/ from 01_2 and the synthetic bags of 01_3 for that dimension)
FEATURES_ROOT = Path('/mnt/Scratch/phd_streamembed_features')           # STEP 2: Folder with level_1/ (original vectors), the pca_<d>/ folders and synthetic/
FEATURES_DIR = FEATURES_ROOT / f'pca_{DIMENSION}'                       # STEP 3: Real PCA vectors (stop, held-out and Testing slides)
FEATURES_FULL_DIR = FEATURES_ROOT / 'level_1'                           # STEP 4: Original 1536-d vectors (only the Testing slides are read, for the comparisons of section 8)
SYNTHETIC_ROOT = FEATURES_ROOT / 'synthetic'                            # STEP 5: Root of the synthetic bags of notebook 01_3
SYNTHETIC_METHOD = 'gaussian'                                           # STEP 6: Generation method (folder name)
SYNTHETIC_LABELS_FILE = Path('labels/BRACS_synthetic.csv')              # STEP 7: Label sheet of the synthetic slides (written by 01_3)
K_VALUES = [5, 10, 25, 50]                                              # STEP 8: Numbers of clusters of the synthetic sets that the search can choose from
SCORING = 'cv'                                                          # STEP 9: 'cv' (all held-out slides, comparable with 02_3, optimistic) or 'validation' (held-out official Validation slides unseen by the generator)
CACHE_IN_RAM = True                                                     # STEP 10: Keep the synthetic bags that were read in memory (float16)
MODEL_NAME = 'UNI2-h'                                                   # STEP 11: Subfolder with the per-slide features
LABELS_FILE = Path('labels/BRACS.xlsx')                                 # STEP 12: Slide labels, patients and official split
MODEL_DIR = Path('classifier/final')                                    # STEP 13: Folder of the saved classifiers
SAVE_MODELS = True                                                      # STEP 14: Save the ensemble as ensemble_<date-time>.pt
SEED = 0                                                                # STEP 15: Seed of every random choice
N_FOLDS, EARLY_STOP_FRACTION = 5, 0.2                                   # STEP 16: Outer folds; share of the non-held-out slides used to pick the epoch (patient-grouped); as in 02_3
EPOCHS, PATIENCE = 100, 5                                               # STEP 17: Maximum epochs; early-stopping patience (counted after the warm-up)
WEIGHTING, TILE_DROPOUT, BAG_WEIGHT = 'sqrt', 0.5, 0.7                  # STEP 18: Class weighting, share of tiles removed per training bag, weight of the slide loss in CLAM-MB
N_TRIALS = 20                                                           # STEP 19: Trials of the Bayesian optimisation (finished trials; a restart continues the study)
STUDY_DIR = Path('outputs/02_4_tuning')                                 # STEP 20: Folder of the study (SQLite), the trials table
STUDY_NAME = f'three_class_syn_pca{DIMENSION}_{SCORING}'                # STEP 21: Name of the study inside the database (one study per dimension and scoring)
MIN_CLASS_RECALL = 0.40                                                 # STEP 22: Smallest out-of-fold recall that every class must have (a float, a dict per class, or None)
N_CONFIRM, REPEATS = 2, 2                                               # STEP 23: Settings confirmed with new seeds; repeats per fold (models of the ensemble = folds x repeats)
N_BOOT = 1000                                                           # STEP 24: Bootstrap resamples (patients) for the intervals
THREE_MAPPING = {'N': 'Non-cancer', 'PB': 'Non-cancer', 'UDH': 'Non-cancer', 'FEA': 'Non-cancer',
                 'ADH': 'Non-invasive atypia/neoplasia', 'DCIS': 'Non-invasive atypia/neoplasia', 'IC': 'Invasive carcinoma'}   # STEP 25: BRACS class -> group (order = model outputs)
TIERS_3 = {'non-cancer': ['Non-cancer'], 'lesion': ['Non-invasive atypia/neoplasia', 'Invasive carcinoma']}                      # STEP 26: Tiers of the coarse auxiliary loss
SPACE = {**DEFAULT_SPACE, 'syn_k': K_VALUES}                            # STEP 27: Search ranges: (low, high) numeric, list = categorical; syn_k = clusters of the synthetic bags
BASELINE = dict(lr=2e-4, weight_decay=1e-1, dropout=0.25, coarse_weight=0.0, k_sample=8, max_tiles=2048, syn_k=K_VALUES[len(K_VALUES) // 2])   # STEP 28: First trial: the setting of notebook 02_1
FIXED = dict(epochs=EPOCHS, patience=PATIENCE, tile_dropout=TILE_DROPOUT, bag_weight=BAG_WEIGHT, weighted_val_loss=True, tiers=TIERS_3)   # STEP 29: train_model arguments that are not searched
STUDY_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)

## 3. Data

### 3.1 Real slides and tile embeddings

The real bags (PCA scores of the UNI2-h CLS tokens, `DIMENSION` components, from notebook 01_2) are loaded once. They serve as the early-stopping, held-out and Testing slides; the slide table holds the official split of BRACS. The dimension is checked against `DIMENSION`.

In [5]:
# STEP 1: Label table with the official split and the bags of tile embeddings (float16)
table = build_slide_table(LABELS_FILE)
data = load_bags(FEATURES_DIR, MODEL_NAME, table)

# STEP 2: The vectors must have the chosen dimension
assert data['dim'] == DIMENSION, f"the files in {FEATURES_DIR} have {data['dim']} components, not {DIMENSION}"
print('PCA dimension:', data['dim'])

Loading bags:   0%|          | 0/547 [00:00<?, ?it/s]

Slides with features: 547 of 547  |  tiles 5,748,673  |  tiles per slide: median 9441, min 9, max 35287  |  dim 256
Per set (with features / labelled): Training 395/395, Validation 65/65, Testing 87/87
Tiling in the files: level 1, tile 448, stride 448
PCA dimension: 256


### 3.2 The three classes

Non-cancer groups N, PB, UDH and FEA; ADH and DCIS form one class; IC keeps its own class. The real slides per class and part are listed.

In [6]:
# STEP 1: Relabel the table and the bags for the three classes
table3, classes3 = relabel_table(data['table'], THREE_MAPPING)
data3 = {**data, 'table': table3, 'classes': classes3}

# STEP 2: Slides per class and part
pd.crosstab(table3['label'], table3['set']).reindex(classes3)[['Training', 'Validation', 'Testing']]

set,Training,Validation,Testing
label,,,
Non-cancer,227,36,43
Non-invasive atypia/neoplasia,68,17,24
Invasive carcinoma,100,12,20


### 3.3 Cross-validation folds

The official Training and Validation slides are pooled and divided into the same patient-grouped, class-stratified folds as in notebook 02_3 (same `N_FOLDS`, `EARLY_STOP_FRACTION` and `SEED`), so the held-out slides, the early-stopping slides and the Testing slides are identical and the results are paired with those of 02_3. The Testing slides are in no fold. In this notebook the *fit* slides of a fold are not used: the model is fitted on synthetic slides, and the early-stopping slides of the fold select the epoch.

In [7]:
# STEP 1: The folds (the Testing slides are left out of every fold)
folds = make_cv_folds(table3, N_FOLDS, SEED, EARLY_STOP_FRACTION)

# STEP 2: Slides and patients per part, and the held-out slides per class
fold_summary(folds, classes3)

,fold,fit,early stopping,held out,held-out patients,Non-cancer,Non-invasive atypia/neoplasia,Invasive carcinoma
0,0,293,74,93,34,53,17,23
1,1,295,72,93,33,53,17,23
2,2,295,73,92,33,53,17,22
3,3,294,75,91,29,52,17,22
4,4,297,72,91,29,52,17,22


### 3.4 Synthetic bags

The synthetic slides of every `k` in `K_VALUES` are read from the label sheet of notebook 01_3 and given the three-class labels of `THREE_MAPPING`. Every file is checked for its dimension and a few for finite values; no synthetic slide may carry a real slide identifier. In the fit data, each synthetic slide is a separate unit (no patient), and the class weights and the standardisation of the model come from these slides only. The class balance is set by the generator (the same number of synthetic bags per BRACS class), so it differs from the real Training set; the class-weighted loss of 02_3 is retained. The tables give the synthetic slides per class and the tiles per slide for each `k`.

In [8]:
# STEP 1: The synthetic bags of every k (lazy; they are read in training), with the three-class labels
synthetic_sets = load_synthetic_sets(SYNTHETIC_LABELS_FILE, SYNTHETIC_ROOT, SYNTHETIC_METHOD, DIMENSION, K_VALUES, THREE_MAPPING, classes3, CACHE_IN_RAM)

# STEP 2: Synthetic slides per class, and tiles per synthetic slide
display(pd.DataFrame({k: s['table']['label'].value_counts().reindex(classes3) for k, s in synthetic_sets.items()}).rename_axis('slides per class', axis=1))
display(pd.DataFrame({k: synthetic_tile_counts(s).describe() for k, s in synthetic_sets.items()}).T.round(0).rename_axis('tiles per slide'))

slides per class,5,10,25,50
label,,,,
Non-cancer,400,400,400,400
Non-invasive atypia/neoplasia,200,200,200,200
Invasive carcinoma,100,100,100,100


,count,mean,std,min,25%,50%,75%,max
tiles per slide,,,,,,,,
5,700.0,10737.0,4491.0,997.0,7714.0,10448.0,13250.0,27513.0
10,700.0,10846.0,4475.0,964.0,7706.0,10698.0,13400.0,25242.0
25,700.0,10649.0,4338.0,861.0,7822.0,10344.0,13272.0,28493.0
50,700.0,10739.0,4607.0,471.0,7458.0,10526.0,13488.0,25880.0


### 3.5 Slides of the score

With `SCORING = 'cv'` every held-out slide of the pool is scored. With `'validation'` only the official Validation slides of patients that have no Training slide are scored, because the generator of notebook 01_3 was estimated on all official Training slides (`utils.synthetic_training.validation_mask`). The number of scored slides per class is listed.

In [9]:
# STEP 1: The pool slides (order of the out-of-fold arrays) and the scored ones
pool_slides = folds[0]['table']['slide'].tolist()
SCORE_MASK = validation_mask(table3, pool_slides) if SCORING == 'validation' else np.ones(len(pool_slides), dtype=bool)
VALIDATION_MASK = validation_mask(table3, pool_slides)                  # always kept: the clean subset, for the comparison of the two scorings in section 7

# STEP 2: Scored slides per class, for the chosen scoring and for the clean subset
labels_pool = folds[0]['table']['label'].to_numpy()
pd.DataFrame({'scored (SCORING)': pd.Series(labels_pool[SCORE_MASK]).value_counts(), 'clean subset': pd.Series(labels_pool[VALIDATION_MASK]).value_counts()}).reindex(classes3).fillna(0).astype(int)

,scored (SCORING),clean subset
Non-cancer,263,34
Non-invasive atypia/neoplasia,85,17
Invasive carcinoma,112,12


## 4. Bayesian optimisation

One trial trains `N_FOLDS` models, one per fold, with a sampled setting and a sampled number of clusters `syn_k`: each model is fitted on all synthetic slides of that `k`, its epoch is selected on the real early-stopping slides of its fold, and it predicts the real held-out slides of the fold. The value of the trial is the macro-averaged F1 of the out-of-fold predictions of the slides chosen by `SCORING`. The optimiser is the tree-structured Parzen estimator of optuna (Bergstra et al., 2011); the study is stored in SQLite, so a restart continues it.

In [ ]:
# STEP 1: The objective (one trial = one cross-validation on the synthetic bags of the sampled k) and the study
objective = synthetic_objective_factory(data3, folds, synthetic_sets, DEVICE, FIXED, WEIGHTING, SEED, SPACE, SCORING, SCORE_MASK)
study = run_study(STUDY_NAME, f'sqlite:///{STUDY_DIR / "study.db"}', objective, N_TRIALS, SEED, BASELINE)

[I 2026-11-17 11:38:27,615] A new study created in RDB with name: three_class_syn_pca256_cv


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-11-17 11:39:54,066] Trial 0 finished with value: 0.6103109757035806 and parameters: {'lr': 0.0002, 'weight_decay': 0.1, 'dropout': 0.25, 'coarse_weight': 0.0, 'k_sample': 8, 'max_tiles': 2048, 'syn_k': 25}. Best is trial 0 with value: 0.6103109757035806.


/home/guisazevedo/Projects/01_accamargo/phd_streamembed/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


[I 2026-11-17 11:42:10,445] Trial 1 finished with value: 0.6334177391160135 and parameters: {'lr': 0.000258817021493792, 'weight_decay': 0.05910317077425309, 'dropout': 0.40138168803582197, 'coarse_weight': 0.3, 'k_sample': 8, 'max_tiles': 8192, 'syn_k': 25}. Best is trial 1 with value: 0.6334177391160135.


### 4.1 Trials

Every finished trial with its parameters (including `syn_k`), the out-of-fold macro-F1 of the scored slides, the out-of-fold recall and F1 of each class, the standard deviation over folds and the minutes it took. Trial 0 is the setting of notebook 02_1.

In [ ]:
# STEP 1: The trials table, best first (ties: lower log-loss)
trials = trials_table(study)
trials.to_csv(STUDY_DIR / f'trials_{STUDY_NAME}.csv', index=False)
trials.round(3)

### 4.2 Optimisation history and importance

In [ ]:
# STEP 1: Figure
fig_tuning = plot_tuning(study)

### 4.3 Number of clusters

The best and the median macro-F1 of the trials for each `syn_k`. The trials differ in their other parameters as well, so the table shows association, not the effect of the number of clusters alone.

In [ ]:
# STEP 1: Macro-F1 of the trials by number of clusters of the synthetic bags
trials.groupby('syn_k')['macro_f1'].agg(trials='size', best='max', median='median').round(3)

### 4.3 Best recall per class

The highest out-of-fold recall that any trial reached for each class. A recall floor above these values would remove every trial.

In [ ]:
# STEP 1: Best out-of-fold recall of each class over the trials
trials[[f'recall_{c}' for c in classes3]].max().round(3).to_frame('best recall').T

## 5. Selection

The final setting is chosen without the Testing set, in three steps: settings in which some class has an out-of-fold recall below `MIN_CLASS_RECALL` are removed; the best pooled macro-F1 is found among the rest; and among the settings within one standard error of it (the standard error of the fold macro-F1 values of the scored slides), the one with the higher dropout, then the higher weight decay is taken (one-standard-error rule, Hastie et al., 2009). Macro-F1 gives every class the same weight, which suits the smaller classes; sensitivity alone is not used, since it is maximised by assigning every slide to the class.

In [ ]:
# STEP 1: Chosen setting and the candidates within one standard error of the best
chosen, candidates = select_setting(trials, MIN_CLASS_RECALL, classes3)
display(candidates.round(3))
print('Chosen: trial', int(chosen['trial']), setting_with_k(chosen))

## 6. Confirmation with new seeds

The chosen setting and the next best settings are trained again with `REPEATS` repeats per fold and seeds that the search did not use, because the best value of many trials is optimistic. The same selection rule is then applied to these scores, and the final setting is the one it chooses.

In [ ]:
# STEP 1: The settings to confirm: the chosen one, then the best remaining ones
order = [int(chosen['trial'])] + [int(t) for t in eligible_trials(trials, MIN_CLASS_RECALL, classes3)['trial'] if int(t) != int(chosen['trial'])]
confirm_trials = order[:N_CONFIRM]

# STEP 2: Cross-validation of each with REPEATS repeats and new seeds, on the synthetic bags of its k
cvs, rows = {}, []
for number in confirm_trials:
    setting = setting_with_k(trials[trials['trial'] == number].iloc[0])
    cvs[number] = synthetic_cross_validate(data3, folds, synthetic_sets[setting['syn_k']], DEVICE, WEIGHTING, REPEATS, SEED + 5000, **setting_kwargs(setting, FIXED))
    rows.append({'trial': number, **setting, **score_oof(cvs[number], classes3, SCORING, SCORE_MASK)})
confirmation = pd.DataFrame(rows).sort_values(['macro_f1', 'log_loss'], ascending=[False, True]).reset_index(drop=True)
confirmation.round(3)

The same selection rule is applied to the confirmation scores, which come from `REPEATS` models per fold, and gives the final setting and its fold models.

In [ ]:
# STEP 1: The final setting by the same rule
final, final_candidates = select_setting(confirmation, MIN_CLASS_RECALL, classes3)
final_trial, final_setting = int(final['trial']), setting_with_k(final)
final_cv = cvs[final_trial]
print('Final: trial', final_trial, final_setting)

## 7. Ensemble

The ensemble consists of the models of the final setting only: one per fold and repeat, `N_FOLDS` x `REPEATS` models, each trained on a different patient-grouped part of Training + Validation and stopped at its own epoch. A held-out slide is predicted by the models of its own fold and their probabilities are averaged, which gives the out-of-fold estimate below without the Testing set; on the Testing set all models predict every slide. Models of different settings are not mixed, because the same folds ranked the settings. The table compares each single repeat with the average of the repeats; the average over the confirmed settings is shown for comparison. The tables are restricted to the slides of the chosen `SCORING`; the last table gives the out-of-fold macro-F1 of the ensemble for all held-out slides and for the clean subset that the generator never saw, which shows how far the pooled score is optimistic.

In [ ]:
# STEP 1: Single repeats against the ensemble, on the scored out-of-fold slides
per_repeat = [p[SCORE_MASK] for p in oof_by_repeat(final_cv, data3, folds, DEVICE)]
scored_cv = subset_cv(final_cv, classes3, SCORE_MASK)
display(single_vs_ensemble(scored_cv, per_repeat, classes3).round(3))

# STEP 2: For comparison, the average over the confirmed settings (the same folds ranked them, so this is optimistic)
from sklearn.metrics import f1_score
mixed = np.mean([cvs[t]['proba'] for t in confirm_trials], axis=0)[SCORE_MASK]
print('Macro F1 of the ensemble of the final setting:', round(score_cv(scored_cv, classes3)['macro_f1'], 3),
      '| of the average over the confirmed settings:', round(f1_score(scored_cv['y'], mixed.argmax(1), average='macro'), 3))

# STEP 3: The two scorings of the final ensemble
display(pd.DataFrame({'all held-out slides (cv)': score_oof(final_cv, classes3, 'cv'), 'clean subset (validation)': score_oof(final_cv, classes3, 'validation', VALIDATION_MASK)}).loc[['macro_f1', 'log_loss'] + [f'recall_{c}' for c in classes3]].round(3).T)

## 8. Evaluation on the Testing set

The Testing labels are read here for the first time. Every model of the ensemble predicts every Testing slide and the probabilities are averaged; the class with the highest probability is the prediction. For each class the measures are computed one class against the rest (for the macro row, the multi-class accuracy and the unweighted mean of the others), with 95% intervals from a bootstrap that resamples patients.

### 8.1 Confusion matrix, ROC curves and measures

In [ ]:
# STEP 1: Testing slides, in table order, and the ensemble probabilities
test = table3[table3['set'] == 'Testing'].reset_index(drop=True)
y_test = np.array([classes3.index(l) for l in test['label']])
patients_test = test['patient'].to_numpy()
proba_test = predict_ensemble(final_cv['runs'], data3, test['slide'].tolist(), DEVICE)['proba']

# STEP 2: Measures with patient-bootstrap intervals, figure and table
NAME = f'Synthetic-trained PCA-{DIMENSION} 3-class ensemble (02_4)'
evaluation = evaluate_predictions(y_test, proba_test, classes3)
views = {NAME: class_views(y_test, proba_test, classes3)}
intervals = {NAME: bootstrap_views(y_test, proba_test, patients_test, classes3, N_BOOT, SEED)}
fig_test = plot_final_results(NAME, evaluation['confusion'], y_test, proba_test, classes3)
display(format_views(views[NAME], intervals[NAME]))

### 8.2 Original vectors of the Testing slides

The comparison models of notebooks 02_1 and 02_2 were trained on the 1536-dimensional vectors, so they must score the original vectors of the Testing slides; only these slides are read at full dimension. The slides are the same as above, in the same order.

In [ ]:
# STEP 1: The original CLS vectors of the Testing slides, with the three-class labels
full = load_bags(FEATURES_FULL_DIR, MODEL_NAME, data['table'][data['table']['set'] == 'Testing'])
full_table3, _ = relabel_table(full['table'], THREE_MAPPING)
data_full3 = {**full, 'table': full_table3, 'classes': classes3}

# STEP 2: Same slides, same order, as the PCA ensemble predicts
assert full['table']['slide'].tolist() == test['slide'].tolist(), 'the Testing slides differ between the PCA and the original vectors'
print('Original vector dimension:', full['dim'])

### 8.3 Comparison with the PCA ensemble of notebook 02_3

The ensemble saved by notebook 02_3 (`classifier/final/noncancer_noninvasive_invasive_pca<DIMENSION>/`) was tuned and trained with the same classes, folds and search on the **real** PCA vectors of the same dimension. Both ensembles predict the same Testing slides (the PCA vectors) and the difference (synthetic-trained minus real-trained) comes with a 95% interval from a paired patient bootstrap, in which both models are scored on the same resamples. An interval that contains zero is not distinguishable from no change at this sample size (87 slides, 31 patients). The comparison isolates the effect of replacing the real fit slides by synthetic ones; the two searches are separate, and 02_4 also searches the number of clusters.

In [ ]:
# STEP 1: The ensemble of notebook 02_3 (the latest file of its folder), on the same slides and the same PCA vectors
pca_files = sorted((MODEL_DIR / f'noncancer_noninvasive_invasive_pca{DIMENSION}').glob('ensemble_*.pt'))
if not pca_files:
    print('No ensemble in', MODEL_DIR / f'noncancer_noninvasive_invasive_pca{DIMENSION}', ': run notebook 02_3 with this DIMENSION first. Comparison skipped.')
else:
    pca_ens = load_ensemble(pca_files[-1], DEVICE)
    assert pca_ens['classes'] == classes3, 'The two ensembles must have the same classes in the same order'
    proba_pca = predict_ensemble(pca_ens['runs'], data3, test['slide'].tolist(), DEVICE)['proba']

    # STEP 2: Measures of both, and the paired bootstrap of the difference
    views[f'Real PCA-{DIMENSION} ensemble (02_3)'] = class_views(y_test, proba_pca, classes3)
    display(summary_across_sets(views).round(3))
    differences = paired_bootstrap_views(y_test, proba_test, proba_test.argmax(1), proba_pca, proba_pca.argmax(1), patients_test, classes3, N_BOOT, SEED)
    display(format_differences(differences))

### 8.4 Comparison with the direct three-class model of notebook 02_1

The model saved by section 10 of notebook 02_1 (`classifier/final/noncancer_adhdcis_ic/`, classes Non-cancer, ADH&DCIS and IC, the same slides in the same groups; trained on the 1536-dimensional vectors) predicts the same Testing slides. The difference (synthetic-trained ensemble minus that model) comes with a 95% interval from a paired patient bootstrap, in which both models are scored on the same resamples. An interval that contains zero is not distinguishable from no change at this sample size (87 slides, 31 patients).

In [ ]:
# STEP 1: The previous direct three-class model (the latest file of its folder), on the same slides
previous_files = sorted((MODEL_DIR / 'noncancer_adhdcis_ic').glob('classifier_model_*.pt'))
if not previous_files:
    print('No classifier in', MODEL_DIR / 'noncancer_adhdcis_ic', ': run section 10 of notebook 02_1 first. Comparison skipped.')
else:
    previous = load_classifier(previous_files[-1], DEVICE)
    assert len(previous['classes']) == len(classes3), 'The two models must have the same number of classes'
    assert previous['classes'][0] == 'Non-cancer' and previous['classes'][-1] == 'IC', 'The class order of the two models must be the same (the group names differ)'
    proba_previous = predict(previous['model'], data_full3, test['slide'].tolist(), DEVICE, previous['scaler'])['proba']

    # STEP 2: Measures of both, and the paired bootstrap of the difference
    views['Direct 3-class, 1536-d (02_1)'] = class_views(y_test, proba_previous, classes3)
    display(summary_across_sets(views).round(3))
    differences = paired_bootstrap_views(y_test, proba_test, proba_test.argmax(1), proba_previous, proba_previous.argmax(1), patients_test, classes3, N_BOOT, SEED)
    display(format_differences(differences))

### 8.5 Comparison with the tuned ensemble of notebook 02_2

The ensemble saved by notebook 02_2 (`classifier/final/noncancer_noninvasive_invasive_tuned/`) uses the same classes, folds and search, but is trained on the 1536-dimensional vectors. Both ensembles predict the same Testing slides, and the difference (synthetic-trained PCA minus real-trained 1536-d) comes with a 95% interval from the paired patient bootstrap. The comparison combines the effect of the dimension reduction and of the synthetic fit data, together with the setting that each search chose; the searches are separate, so it does not isolate either factor.

In [ ]:
# STEP 1: The ensemble of notebook 02_2 (the latest file of its folder), on the same slides
tuned_files = sorted((MODEL_DIR / 'noncancer_noninvasive_invasive_tuned').glob('ensemble_*.pt'))
if not tuned_files:
    print('No ensemble in', MODEL_DIR / 'noncancer_noninvasive_invasive_tuned', ': run notebook 02_2 first. Comparison skipped.')
else:
    tuned = load_ensemble(tuned_files[-1], DEVICE)
    assert tuned['classes'] == classes3, 'The two ensembles must have the same classes in the same order'
    proba_tuned = predict_ensemble(tuned['runs'], data_full3, test['slide'].tolist(), DEVICE)['proba']

    # STEP 2: Measures of both, and the paired bootstrap of the difference
    views['Tuned, 1536-d (02_2)'] = class_views(y_test, proba_tuned, classes3)
    display(summary_across_sets(views).round(3))
    differences = paired_bootstrap_views(y_test, proba_test, proba_test.argmax(1), proba_tuned, proba_tuned.argmax(1), patients_test, classes3, N_BOOT, SEED)
    display(format_differences(differences))

### 8.6 Summary of the compared results

All models on the same Testing slides: the macro row of each model (accuracy, sensitivity, specificity, precision, F1 and AUROC; one class against the rest, then averaged without weights). Differences with intervals are given in sections 8.3 to 8.5; the Testing set has few slides and patients, so small differences are not resolved.

In [ ]:
# STEP 1: The measures of every model that was available
display(summary_across_sets(views).round(3))

## 9. Save

The ensemble is written to one file with the weights, scaler and settings of every model (`utils.mil_tuning.save_ensemble`; `load_ensemble` returns the models for `predict_ensemble`). The configuration records the searched ranges, the final setting (including the number of clusters), the scoring, the study and the metadata of the synthetic set that the ensemble was fitted on.

In [ ]:
# STEP 1: What is needed to reproduce and to reuse the ensemble
syn_folder = synthetic_sets[final_setting['syn_k']]['folder']
syn_metadata = json.loads((syn_folder / 'metadata.json').read_text()) if (syn_folder / 'metadata.json').exists() else {}
config = dict(
    pca=dict(dimension=DIMENSION, vectors=str(FEATURES_DIR / MODEL_NAME), metadata=str(FEATURES_DIR / 'metadata.json')),
    synthetic=dict(root=str(SYNTHETIC_ROOT), method=SYNTHETIC_METHOD, labels_file=str(SYNTHETIC_LABELS_FILE), k_values=K_VALUES, syn_k=final_setting['syn_k'], folder=str(syn_folder), metadata=syn_metadata),
    scoring=SCORING, seed=SEED, n_folds=N_FOLDS, repeats=REPEATS, early_stop_fraction=EARLY_STOP_FRACTION, weighting=WEIGHTING, fixed={k: v for k, v in FIXED.items() if k != 'tiers'}, tiers=TIERS_3,
    final_trial=final_trial, final_setting=final_setting, space={k: list(v) if isinstance(v, (tuple, list)) else v for k, v in SPACE.items()}, n_trials=N_TRIALS, min_class_recall=MIN_CLASS_RECALL,
    mapping=THREE_MAPPING, classes=classes3, features_dir=str(FEATURES_DIR), model_name=MODEL_NAME, tile_params=data['tile_params'], device=device_description(DEVICE),
    versions=dict(torch=torch.__version__, sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__, optuna=optuna.__version__), dropped_slides=data['missing'],
)

# STEP 2: Save the models of the ensemble
if SAVE_MODELS:
    path = save_ensemble(final_cv['runs'], MODEL_DIR / f'noncancer_noninvasive_invasive_synpca{DIMENSION}', config)
    print('Ensemble of', len(final_cv['runs']), 'models saved to', path.resolve())
else:
    print('SAVE_MODELS is False: nothing was written.')